# Phase 1: Data + Training (Custom Dataset to YOLOv8)
Run this notebook in Google Colab with a **T4 GPU** enabled (`Runtime -> Change runtime type -> T4 GPU`).

This notebook trains the YOLOv8n model on the pre-compiled `Final_Training_Dataset.zip` covering all 5 requested objects:
1. Shipwreck
2. Aircraft
3. Pipe
4. Cylinder
5. Ghost Net

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Install dependencies
!pip install ultralytics pyyaml

In [ ]:
# 3. Copy Dataset from Drive to LOCAL Colab disk and Unzip
# IMPORTANT: Ensure 'Final_Training_Dataset.zip' is uploaded to the root of your Google Drive.

import os

drive_zip_path = "/content/drive/MyDrive/Final_Training_Dataset.zip"
local_extract_path = "/content/"

if os.path.exists(drive_zip_path):
    print("Copying zip from Drive to local Colab disk...")
    !cp "{drive_zip_path}" /content/Final_Training_Dataset.zip

    print("Unzipping to local disk...")
    !unzip -q /content/Final_Training_Dataset.zip -d {local_extract_path}
    print("Dataset successfully extracted to /content/Final_Training_Dataset/")
else:
    print(f"ERROR: Could not find {drive_zip_path}. Please upload it to your Drive first!")

### IMPORTANT: Data Format Check
Our `Final_Training_Dataset.zip` is ALREADY in YOLO format with `.txt` labels and a `data.yaml` file generated by our local compilation script. 
We can skip any XML-to-YOLO conversion steps and train directly!

In [ ]:
# 4. Train YOLOv8n on GPU
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data="/content/Final_Training_Dataset/data.yaml",
    epochs=100,
    imgsz=640,
    project="/content/sonar_detection",
    name="yolov8n_sonar",
    cache=True, # <--- Add this flag!
    device=0
)

In [ ]:
# 5. Export to ONNX
best_model = YOLO("/content/sonar_detection/yolov8n_sonar/weights/best.pt")
export_path = best_model.export(format="onnx")
print(f"Exported ONNX model to: {export_path}")

In [ ]:
# 6. Save outputs to Google Drive (so they persist after Colab disconnects)
import shutil
import os

drive_save_dir = "/content/drive/MyDrive/GhostNetSonar_Weights"
os.makedirs(drive_save_dir, exist_ok=True)

!cp /content/sonar_detection/yolov8n_sonar/weights/best.pt {drive_save_dir}/
!cp /content/sonar_detection/yolov8n_sonar/weights/best.onnx {drive_save_dir}/
!cp /content/Final_Training_Dataset/data.yaml {drive_save_dir}/
print("Saved weights and data.yaml to Google Drive!")